## Главная задача

Из накопительных показаний получить **суточное потребление** (расход за день =
разница показаний между соседними днями) и найти в потреблении аномалии.

## Суточные потребления

In [34]:
import pandas as pd

df_pokazaniya = pd.read_csv(
    "pokazaniya.csv",
    sep=";",
    encoding="utf-8-sig",
    dtype={"anon_id": str},
    parse_dates=["data"],
)

df_pokazaniya = df_pokazaniya.sort_values(["anon_id", "data"])

df_pokazaniya["dni_mezhdu"] = df_pokazaniya.groupby("anon_id")["data"].diff().dt.days
df_pokazaniya["raznica"] = df_pokazaniya.groupby("anon_id")["pokazanie"].diff().round(3)

# Расходом именно за одни сутки считаем разницу только соседних дат.
df_pokazaniya["rashod_za_sutki"] = df_pokazaniya["raznica"].where(df_pokazaniya["dni_mezhdu"] == 1)

In [35]:
df_pokazaniya.to_csv('pokazanaiya_res.csv', sep=";", index=False, encoding="utf-8-sig")

## Что найти (аномалии потребления)

Для каждого пункта: ваш критерий (порог) + сколько приборов + список/примеры
самых ярких `anon_id`.

## 1. **Отрицательный расход** — показание уменьшилось (счётчик «пошёл назад»).
   Что это может означать?
   накопительное показание обычно должно увеличиваться или оставаться прежним. Его уменьшение может быть связано со сбросом счётчика, заменой прибора, ошибкой передачи данных, неисправностью или обратным движением счётчика. По отрицательной разнице нельзя определить точную причину — нужно проверить события «сброс» и историю прибора.

In [38]:
# Строки, где показание уменьшилось
df_minus_rasxod = df_pokazaniya[
    df_pokazaniya["raznica"] < 0
]

# Количество уникальных приборов
print("Приборов:", df_minus_rasxod["anon_id"].nunique())

# Количество случаев уменьшения
print("Случаев:", len(df_minus_rasxod))

# Количество падений показаний у каждого прибора
top_minus = (
    df_minus_rasxod.groupby("anon_id")
    .size()
    .sort_values(ascending=False)
    .head(5)
    .rename("kolichestvo_padeniy")
)

print('В первую очередь стоит проверить приборы приведенные ниже у них самое большое количество падений')
display(top_minus.to_frame())



Приборов: 102
Случаев: 269
В первую очередь стоит проверить приборы приведенные ниже у них самое большое количество падений


,kolichestvo_padeniy
anon_id,
2030325,13
7931977,12
3917499,10
8190741,10
7293232,8


## 2. **Всплески** — аномально большой расход за сутки. Дайте и абсолютный порог,
   и относительный (например, в N раз больше обычного для этого прибора).

Критерии:
- Абсолютный: расход больше 1 м³ за сутки — 1000 литров.
- Относительный: расход больше обычного для этого прибора в 5 раз, при этом больше 0.1 м³ — чтобы не считать всплеском совсем маленькие изменения.
- Обычный расход: медиана положительных суточных расходов прибора. Например, для значений 0.1, 0.2, 0.3 медиана равна 0.2 м³.

In [42]:
# Обычный порого медиана
df_pokazaniya["obychnyy_rashod"] = (
    df_pokazaniya["rashod_za_sutki"]
    .where(df_pokazaniya["rashod_za_sutki"] > 0)
    .groupby(df_pokazaniya["anon_id"])
    .transform("median")
)

In [48]:
# Абсолютный порог больше 1м^3
mask_abs = df_pokazaniya["rashod_za_sutki"] > 1
df_vspleski_abs = df_pokazaniya[mask_abs]

print("Приборов:", df_vspleski_abs["anon_id"].nunique())
print("Случаев:", len(df_vspleski_abs))

display(
    df_vspleski_abs.groupby("anon_id")
    .size()
    .sort_values(ascending=False)
    .head(5)
    .rename("kolichestvo_1m3")
    .to_frame()
)

Приборов: 367
Случаев: 7697


,kolichestvo_1m3
anon_id,
1606847,282
7199412,279
5754693,254
1648954,241
3161059,239


In [47]:
# Относительный порог превышающий в 5 раз (можно считать как всплески)
mask_rel = (
    (df_pokazaniya["rashod_za_sutki"]
     > 5 * df_pokazaniya["obychnyy_rashod"])
    & (df_pokazaniya["rashod_za_sutki"] > 0.1)
)

df_vspleski_rel = df_pokazaniya[mask_rel]

print("Приборов:", df_vspleski_rel["anon_id"].nunique())
print("Случаев:", len(df_vspleski_rel))

display(
    df_vspleski_rel.groupby("anon_id")
    .size()
    .sort_values(ascending=False)
    .head(5)
    .rename("kolichestvo_vspleskov")
    .to_frame()
)

Приборов: 750
Случаев: 4120


,kolichestvo_vspleskov
anon_id,
2158929,137
3190895,105
6675553,93
3036541,76
7626821,52


Всплеском считаем то что одновременно превышает относительный порог и абсолютный порого(для некоторых 1м3 может быть практически нормой)

In [45]:
df_vspleski_both = df_pokazaniya[mask_abs & mask_rel]

print("Приборов:", df_vspleski_both["anon_id"].nunique())
print("Случаев:", len(df_vspleski_both))

display(
    df_vspleski_both.groupby("anon_id")
    .size()
    .sort_values(ascending=False)
    .head(5)
    .rename("kolichestvo_vspleskov")
    .to_frame()
)

Приборов: 210
Случаев: 917


,kolichestvo_vspleskov
anon_id,
3036541,57
2030325,48
3743114,35
1278937,29
8727125,24


## 3 **Нереальные значения** — физически невозможный расход (десятки и тысячи м³ в сутки). Найдите такие приборы.

In [50]:
df_nereal = df_pokazaniya[
    df_pokazaniya["rashod_za_sutki"] > 10
]

print("Приборов:", df_nereal["anon_id"].nunique())
print("Случаев:", len(df_nereal))

# Самые большие расходы
print(
    df_nereal.nlargest(10, "rashod_za_sutki")[
        ["anon_id", "data", "rashod_za_sutki"]
    ].to_string(index=False)
)

Приборов: 63
Случаев: 379
anon_id       data  rashod_za_sutki
2491162 2026-07-09      3005503.868
8908023 2026-09-07      2684039.253
2159987 2025-10-17         2834.770
7891133 2025-10-17         2124.960
8973572 2025-10-17          608.136
5359103 2026-07-05          530.317
2030325 2026-07-20          425.916
4964924 2026-09-20          401.047
7247006 2026-09-29          364.053
8498147 2026-09-29          363.971


## 4 **Утечка** — непрерывное потребление без «нулевых» суток длительное время
   (прибор никогда не «отдыхает»). Ваш критерий и сколько.

In [56]:
# Положительный расход за сутки
mask = df_pokazaniya["rashod_za_sutki"] > 0

# Новая серия при нарушении условия или смене прибора
seriya = (
    ~mask
    | df_pokazaniya["anon_id"].ne(df_pokazaniya["anon_id"].shift())
).cumsum()
# Длина каждой серии положительного расхода
serii = (
    df_pokazaniya[mask]
    .groupby([df_pokazaniya["anon_id"], seriya])
    .size()
)

# Серии длительностью не менее 180 суток
utechki = serii[serii >= 180]
# Самая длинная подходящая серия каждого прибора
pribory_utechki = (
    utechki.groupby("anon_id")
    .max()
    .sort_values(ascending=False)
)
print("Приборов:", len(pribory_utechki))

print(
    pribory_utechki.head(5)
    .rename("sutok_podryad")
    .reset_index()
    .to_string(index=False)
)

Приборов: 79
anon_id  sutok_podryad
1968874            365
7496751            364
2931056            364
6554606            353
1282534            332


## 5 **Нулевое потребление** длительно, хотя прибор продолжает слать пакеты (пустая квартира / застрявший счётчик). Порог и количество.

In [58]:
# Нулевой суточный расход, но пакеты приходят
mask_zero = (
    (df_pokazaniya["rashod_za_sutki"] == 0)
    & (df_pokazaniya["paketov_za_sutki"] > 0)
)

# Номера серий
seriya_zero = (
    ~mask_zero
    | df_pokazaniya["anon_id"].ne(df_pokazaniya["anon_id"].shift())
).cumsum()

# Длина каждой серии
serii_zero = (
    df_pokazaniya[mask_zero]
    .groupby([df_pokazaniya["anon_id"], seriya_zero])
    .size()
)

# Серии от 30 суток
dlitelnyy_zero = serii_zero[serii_zero >= 30]

# Самая длинная серия каждого прибора
pribory_zero = (
    dlitelnyy_zero.groupby("anon_id")
    .max()
    .sort_values(ascending=False)
)

print("Приборов:", len(pribory_zero))

print(
    pribory_zero.head(5)
    .rename("sutok_bez_rashoda")
    .reset_index()
    .to_string(index=False)
)

Приборов: 694
anon_id  sutok_bez_rashoda
9686922                364
7558835                364
9354769                360
5374718                331
1547698                322


##  6

## 7

## 8